# Diagnosis of the base model

Section 3 (Fig. 3b–c) and Appendix A (Table 4) on Qwen3.5-4B (Base), the model S-Mind-4B starts from:

- **Answers on the two-view questions** of MindCube-Tiny (how the camera moved between two photos): how often Base is
  right, and how often its errors place the second camera on the mirrored side.
- **The viewpoint probe on SpinBench**, with the linear axes of S-Space: whether the state of the two objects is right
  after an external rotation to the queried viewpoint, and whether it is already in the queried frame at the first and
  last mention of the objects in the model's own reasoning; and the model's accuracy split by that last state.

**Needs:** the four benchmarks built (`scripts/data/benchmarks.sh`); [S-Space](https://github.com/MirroS-Lab/S-Space) cloned at
`third_party/s-space` and set up with its `scripts/install.sh` and `scripts/prepare_data.sh` (`analysis/README.md`);
one GPU, about an hour and a half.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"
GPU = "0"

In [ ]:
nb.need("data/processed/mindcube_tiny.jsonl", "third_party/s-space")
base = nb.base_model(SIZE)
out = nb.RUNS / f"{SIZE}-base"
nb.run(["python", "-m", "smind.eval.evaluate", "--model", base, "--out", out / "eval"], out / "eval.log",
       done=out / "eval/viewspatial/summary.json", gpu=GPU)

In [ ]:
SSPY = "third_party/s-space/.envs/qwen36/bin/python"
SSPATH = "third_party/s-space:src"
name, sspace = "base", nb.ROOT / "runs/sspace/base"
log = sspace.parent / f"{name}.log"
nb.run(["python", "analysis/diagnosis/sspace/make_configs.py", "--model", name], log)
nb.run(["bash", "analysis/diagnosis/sspace/run_queue.sh", name], log, done=sspace / "run_queue.done", gpu=GPU,
       env=dict(GPU=GPU))
(sspace / "run_queue.done").touch()
for premise in ("with", "without"):
    nb.run([SSPY, "analysis/diagnosis/sspace/spinbench_native_cot.py", "--model", name, "--premise", premise], log,
           done=sspace / f"native_cot_spinbench_{premise}/summary.json", gpu=GPU, pythonpath=SSPATH)
nb.run(["python", "analysis/diagnosis/sspace/compare.py", "--models", name], log,
       done=sspace.parent / "comparison.json", pythonpath=SSPATH)
done = sspace / "cot_evolution_with_premise/base_axes/summary.json"
for step in ("replay", "score"):
    nb.run([SSPY, "analysis/diagnosis/sspace/cot_evolution.py", step, "--model", name, "--premise", "with"], log,
           done=done, gpu=GPU,
           pythonpath=SSPATH)

In [ ]:
nb.run(["python", "analysis/diagnosis/two_view_errors.py", "--model", f"Base={out / 'eval'}",
        "--sspace", f"Base=runs/sspace/{name}", "--out", out / "two_view.json"], out / "two_view.log",
       done=out / "two_view.json")
tv = nb.read_json(out / "two_view.json")
evo = nb.read_json(done)["stages"]
cmp = nb.read_json(sspace.parent / "comparison.json")
acc = lambda section, key: 100 * cmp[section][key][name]["accuracy"]
rows = {"two-view questions: accuracy": tv["two_view"]["Base"]["acc"][0],
        "two-view questions: mirrored, % of errors": tv["two_view"]["Base"]["mirror_share"][0],
        "EmbSpatial: single-image readout": acc("projection", "embspatial_margin"),
        "S-Space two-view set: readout": acc("projection", "hstar_multiview_selected_layer"),
        "SpinBench (premise): state right after external rotation": acc("projection", "spinbench_with_premise_rotation"),
        "SpinBench (no premise): state right after external rotation": acc("projection", "spinbench_without_premise_rotation"),
        "SpinBench (premise): the model's answers": acc("native_cot", "spinbench_with_premise"),
        "SpinBench (no premise): the model's answers": acc("native_cot", "spinbench_without_premise"),
        "SpinBench (premise): state in the queried frame, first mention": 100 * evo["first"]["direct"]["accuracy"],
        "SpinBench (premise): state in the queried frame, last mention": 100 * evo["last"]["direct"]["accuracy"],
        "SpinBench (premise): accuracy when the last state is in the queried frame": tv["state_answer"]["Base"]["turned"]["acc"][0],
        "SpinBench (premise): accuracy otherwise": tv["state_answer"]["Base"]["not_turned"]["acc"][0]}
nb.compare(rows, dict(zip(rows, [40.88, 92.31, 86.50, 82.50, 97.92, 91.10, 72.22, 74.66, 31.93, 52.94, 85.71, 55.36])))

Base perceives the scene, as the readouts right after an external rotation show, but neither fuses the views nor turns
its state to the queried viewpoint: it answers the two-view questions near chance, nearly all its errors there are
mirror images of the right answer, and its state is in the queried frame for only part of the SpinBench questions, on
which its answers are far more often right.